# Deep Hedging Tutorial

**Phase 7.6: Deep Hedging & Neural Optimal Control**

This tutorial introduces the deep hedging framework for learning optimal hedging strategies using reinforcement learning. We'll cover:

1. **The Hedging Problem**: Why we hedge and what makes it challenging
2. **Classical Delta Hedging**: The Black-Scholes approach and its limitations
3. **Deep Hedging Framework**: Learning optimal policies via risk minimisation
4. **Implementation**: Using QuantStrata's deep hedging module
5. **Comparison**: Delta hedging vs. deep hedging
6. **Analysis**: Understanding when and why deep hedging outperforms

**Prerequisites**: Basic understanding of options, Black-Scholes model, and neural networks.

**References**:
- Bühler et al. (2019) "Deep Hedging"
- Technical documentation: `docs/reference/deep_hedging/theory.md`

## 1. Introduction: The Hedging Problem

### 1.1 Why Do We Hedge?

Suppose you're a derivatives dealer who has **sold** a European call option to a client:

- Strike: K = 100
- Maturity: T = 3 months
- Current spot: S₀ = 100

At maturity, you must pay the client max(S_T - K, 0). If the stock rises to 120, you owe 20. If it falls to 80, you owe nothing.

**The problem**: Your P&L is highly uncertain. You collected a premium (say, 5.50), but your liability could be anywhere from 0 to infinity.

**Hedging** aims to reduce this uncertainty by trading in the underlying asset.

### 1.2 The Ideal: Perfect Replication

In the Black-Scholes world, you can **perfectly replicate** the option payoff by:
1. Holding Δ = ∂V/∂S shares of the underlying at each instant
2. Continuously rebalancing as the delta changes

If you do this perfectly, your hedge portfolio exactly offsets the option liability, and your P&L equals zero (you keep the premium).

### 1.3 The Reality: Why Perfect Hedging Fails

| Assumption | Reality |
|------------|--------|
| Continuous trading | You rebalance at discrete times (daily, hourly) |
| No transaction costs | You pay bid-ask spread on every trade |
| Known volatility | Volatility is stochastic and uncertain |
| Unlimited liquidity | Large trades move the market |

**Result**: Real hedging always incurs **hedging error** and/or **transaction costs**.

### 1.4 The Trade-off

There's a fundamental trade-off:
- **Hedge more frequently** → Less tracking error, but more transaction costs
- **Hedge less frequently** → Lower costs, but more tracking error

**Question**: What's the optimal strategy? Delta hedging is a good heuristic, but is it optimal when we account for costs and discrete rebalancing?

This is where **deep hedging** comes in.

## 2. Setup and Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from typing import Dict, List, Tuple

# Set random seed for reproducibility
np.random.seed(42)

# Plotting style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Setup complete!")

In [ ]:
# Import deep hedging components
from src.deep_hedging import (
    # Configuration and types
    HedgingConfig,
    HedgingState,
    HedgingResult,
    
    # Transaction costs
    ProportionalCost,
    ZeroCost,
    
    # Risk measures
    MeanVarianceRisk,
    CVaRRisk,
    
    # Environments
    GBMHedgingEnv,
    
    # Agents
    DeltaHedgingAgent,
    DeepHedgingAgent,
    NoHedgingAgent,
    MLPPolicy,
    
    # Training and evaluation
    simulate_hedging_batch,
    evaluate_agent,
    compare_agents,
    compute_hedging_metrics,
)

print("Deep hedging module imported successfully!")

## 3. Setting Up the Hedging Environment

### 3.1 The Option We're Hedging

Let's set up a simple scenario:
- **European call option** with strike K = 100
- **Maturity** T = 3 months (0.25 years)
- **Initial spot** S₀ = 100 (ATM)
- **Volatility** σ = 20%
- **Risk-free rate** r = 5%
- **Rebalancing** daily (~63 trading days in 3 months)

In [ ]:
# Configure the hedging scenario
config = HedgingConfig(
    option_type="call",
    strike=100.0,
    maturity=0.25,           # 3 months
    spot_initial=100.0,      # ATM
    volatility=0.20,         # 20% annualised
    risk_free_rate=0.05,     # 5%
    n_steps=63,              # Daily rebalancing (~252/4 trading days)
    notional=1.0,            # 1 unit of the option
)

print("Hedging Configuration:")
print(f"  Option: {config.option_type.upper()} with K={config.strike}")
print(f"  Maturity: {config.maturity*12:.1f} months ({config.n_steps} rebalancing steps)")
print(f"  Spot: S₀={config.spot_initial}")
print(f"  Volatility: σ={config.volatility*100:.0f}%")
print(f"  Time step: Δt={config.dt*252:.1f} trading days")

### 3.2 Transaction Costs

We model transaction costs as **proportional to trade size** (bid-ask spread):

$$C(\Delta\delta, S) = \kappa \cdot S \cdot |\Delta\delta|$$

where κ is the half-spread. For example, a 10 basis point (0.1%) round-trip spread means κ = 5bp = 0.0005.

In [ ]:
# Transaction cost model: 10bp round-trip spread
cost_model = ProportionalCost(spread_bps=10.0)

print(f"Transaction cost model: {cost_model}")
print(f"  Half-spread κ = {cost_model.half_spread*10000:.1f} bps")
print(f"  Example: Trading 1 share at S=100 costs ${cost_model.compute(1.0, 100.0):.4f}")

### 3.3 Creating the Hedging Environment

The `GBMHedgingEnv` simulates:
1. **Market dynamics**: Spot evolves under GBM (Black-Scholes)
2. **Hedging P&L**: Position × spot change − transaction costs
3. **Option settlement**: At maturity, we pay the option payoff

In [ ]:
# Create the hedging environment
env = GBMHedgingEnv(config=config, cost_model=cost_model)

print(f"Hedging Environment: {env}")
print(f"\nInitial BSM Price: ${env._compute_bsm_price():.4f}")

## 4. Classical Delta Hedging

### 4.1 How Delta Hedging Works

At each rebalancing time t, we:
1. **Observe** the current spot S_t
2. **Compute** the BSM delta: Δ_t = ∂V/∂S (from the Black-Scholes formula)
3. **Trade** to hold Δ_t shares of the underlying
4. **Pay** transaction costs on the trade

The `DeltaHedgingAgent` implements this strategy.

In [ ]:
# Create delta hedging agent
delta_agent = DeltaHedgingAgent()

print(f"Agent: {delta_agent}")
print(f"\nDelta hedging simply returns the BSM delta from the state.")

### 4.2 Running a Single Hedging Episode

Let's trace through one hedging episode to see how it works:

In [ ]:
# Run a single episode with delta hedging
state, info = env.reset(seed=42)
delta_agent.reset()

print("Initial state:")
print(f"  Spot: {state.spot:.2f}")
print(f"  Delta (BSM): {state.delta_bs:.4f}")
print(f"  Initial premium: {state.pnl:.4f}")

# Track the episode
spots = [state.spot]
positions = []
pnls = [state.pnl]
deltas = [state.delta_bs]

# Run the episode
step = 0
while True:
    # Get action (BSM delta)
    action = delta_agent.select_action(state)
    positions.append(action)
    
    # Take step
    state, reward, terminated, truncated, step_info = env.step(action)
    
    spots.append(state.spot)
    pnls.append(state.pnl)
    if state.delta_bs is not None:
        deltas.append(state.delta_bs)
    
    step += 1
    if terminated or truncated:
        break

# Get the episode record
episode = env.get_episode()

print(f"\nEpisode completed in {step} steps:")
print(f"  Terminal spot: {spots[-1]:.2f}")
print(f"  Option payoff: {episode.payoff:.4f}")
print(f"  Total costs: {episode.total_cost:.4f}")
print(f"  Terminal P&L: {episode.terminal_pnl:.4f}")

In [ ]:
# Visualise the episode
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Spot price path
ax = axes[0, 0]
ax.plot(spots, 'b-', linewidth=1.5)
ax.axhline(y=config.strike, color='r', linestyle='--', label=f'Strike K={config.strike}')
ax.set_xlabel('Step')
ax.set_ylabel('Spot Price')
ax.set_title('Spot Price Path')
ax.legend()

# Hedge position (delta)
ax = axes[0, 1]
ax.plot(positions, 'g-', linewidth=1.5, label='Actual position')
ax.plot(deltas[:-1], 'b--', linewidth=1, alpha=0.7, label='BSM delta')
ax.set_xlabel('Step')
ax.set_ylabel('Position (Δ)')
ax.set_title('Hedge Position')
ax.legend()

# Cumulative P&L
ax = axes[1, 0]
ax.plot(pnls, 'purple', linewidth=1.5)
ax.axhline(y=0, color='black', linestyle='-', alpha=0.3)
ax.set_xlabel('Step')
ax.set_ylabel('Cumulative P&L')
ax.set_title('Hedging P&L Over Time')

# Transaction costs
ax = axes[1, 1]
ax.bar(range(len(episode.costs)), episode.costs, color='red', alpha=0.7)
ax.set_xlabel('Step')
ax.set_ylabel('Cost')
ax.set_title(f'Transaction Costs (Total: {episode.total_cost:.4f})')

plt.tight_layout()
plt.show()

### 4.3 Evaluating Delta Hedging Over Many Episodes

One episode doesn't tell us much. Let's simulate 1000 episodes and look at the P&L distribution:

In [ ]:
# Evaluate delta hedging over many episodes
delta_result = evaluate_agent(delta_agent, env, n_episodes=1000, seed=42)

# Compute metrics
delta_metrics = compute_hedging_metrics(delta_result.pnl_samples, delta_result.cost_samples)

print("Delta Hedging Results (1000 episodes):")
print(f"  Mean P&L:    {delta_metrics['mean_pnl']:>8.4f}")
print(f"  Std P&L:     {delta_metrics['std_pnl']:>8.4f}")
print(f"  Sharpe:      {delta_metrics['sharpe']:>8.3f}")
print(f"  VaR (95%):   {delta_metrics['var_95']:>8.4f}")
print(f"  CVaR (95%):  {delta_metrics['cvar_95']:>8.4f}")
print(f"  Mean Cost:   {delta_metrics['mean_cost']:>8.4f}")

In [ ]:
# Plot P&L distribution
fig, ax = plt.subplots(figsize=(10, 6))

ax.hist(delta_result.pnl_samples, bins=50, density=True, alpha=0.7, 
        color='blue', edgecolor='black', label='Delta Hedging')
ax.axvline(x=0, color='black', linestyle='-', linewidth=2, label='Break-even')
ax.axvline(x=delta_metrics['mean_pnl'], color='blue', linestyle='--', 
           linewidth=2, label=f"Mean: {delta_metrics['mean_pnl']:.4f}")
ax.axvline(x=delta_metrics['cvar_95'], color='red', linestyle=':', 
           linewidth=2, label=f"CVaR 95%: {delta_metrics['cvar_95']:.4f}")

ax.set_xlabel('Terminal P&L', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Delta Hedging P&L Distribution (1000 episodes)', fontsize=14)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

### 4.4 The Problem with Delta Hedging

Notice that:
1. **Mean P&L is slightly negative** — we're losing money on average due to transaction costs
2. **There's significant variance** — despite hedging, outcomes are uncertain
3. **The tail (CVaR) is worse than the mean** — there are some bad outcomes

This is the cost of **discrete rehedging** and **transaction costs**. Can we do better?

## 5. The Deep Hedging Framework

### 5.1 The Key Insight

Instead of deriving the hedge from a model (BSM), we can **learn** the optimal hedging policy directly by:

1. **Simulating** many hedging episodes
2. **Computing** a risk measure (e.g., variance, CVaR) over the P&L distribution
3. **Optimising** the policy to minimise this risk measure

### 5.2 The Policy Network

We parameterise the hedging policy as a neural network:

$$\delta_t = \pi_\theta(s_t)$$

where:
- $s_t$ = state (spot, time, current position, Greeks, ...)
- $\pi_\theta$ = neural network with parameters $\theta$
- $\delta_t$ = hedge position

### 5.3 The Training Objective

We minimise a **risk measure** applied to the terminal P&L:

$$\min_\theta \rho\left( -\text{P\&L}_T^\theta \right)$$

Common choices:
- **Mean-Variance**: $\rho(L) = \mathbb{E}[L] + \lambda \cdot \text{Var}(L)$
- **CVaR**: Expected loss in worst α% of outcomes
- **Entropic**: Exponential utility certainty equivalent

## 6. Implementing Deep Hedging

### 6.1 Creating the Deep Hedging Agent

In [ ]:
# Create the policy network
# Input: [log_moneyness, time_to_maturity, position, pnl, delta, gamma, vega] = 7 features
policy = MLPPolicy(
    input_dim=7,
    hidden_layers=[64, 64],      # Two hidden layers with 64 units each
    output_dim=1,                # Output: hedge position
    activation="relu",           # ReLU activation
    output_activation="tanh",    # Tanh to bound output to [-1, 1]
    output_scale=1.0,            # Scale output (delta is typically in [0, 1])
)

print(f"Policy Network:")
print(f"  Architecture: {policy.input_dim} → {policy.hidden_layers} → {policy.output_dim}")
print(f"  Parameters: {policy.num_parameters()}")

In [ ]:
# Create the deep hedging agent with mean-variance risk
risk_measure = MeanVarianceRisk(risk_aversion=0.5)

deep_agent = DeepHedgingAgent(
    policy=policy,
    risk_measure=risk_measure,
    learning_rate=0.001,
    include_greeks=True,        # Use BSM Greeks as input features
    normalise_features=True,    # Normalise inputs for better training
)

print(f"Deep Hedging Agent: {deep_agent}")

### 6.2 Testing the Untrained Agent

Before training, the agent outputs random (initialised) actions:

In [ ]:
# Evaluate the untrained deep hedging agent
untrained_result = evaluate_agent(deep_agent, env, n_episodes=1000, seed=42)
untrained_metrics = compute_hedging_metrics(untrained_result.pnl_samples, untrained_result.cost_samples)

print("Untrained Deep Hedging Results (1000 episodes):")
print(f"  Mean P&L:    {untrained_metrics['mean_pnl']:>8.4f}")
print(f"  Std P&L:     {untrained_metrics['std_pnl']:>8.4f}")
print(f"  Sharpe:      {untrained_metrics['sharpe']:>8.3f}")
print(f"\n(Compare to Delta Hedging std: {delta_metrics['std_pnl']:.4f})")

### 6.3 Training the Deep Hedging Agent

Now let's train the agent to minimise the risk measure.

**Note**: The NumPy-based trainer uses finite difference gradient estimation, which is slow but illustrative. For production, use TensorFlow/PyTorch for automatic differentiation.

In [ ]:
from src.deep_hedging.training import HedgingTrainer
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

# Create trainer
trainer = HedgingTrainer(
    agent=deep_agent,
    env=env,
    batch_size=256,
    learning_rate=0.001,
)

print("Training the deep hedging agent...")
print("(This may take a minute with the NumPy-based trainer)")

In [ ]:
# Train for a few epochs (increase for better results)
# Note: The NumPy trainer is for illustration; use TensorFlow/PyTorch for real training
training_result = trainer.train(
    n_epochs=20,
    verbose=1,
    log_every=5,
)

print(f"\nTraining completed!")
print(f"  Final loss: {training_result['final_loss']:.4f}")
print(f"  Training time: {training_result['training_time_seconds']:.1f}s")

### 6.4 Evaluating the Trained Agent

In [ ]:
# Evaluate the trained deep hedging agent
deep_result = evaluate_agent(deep_agent, env, n_episodes=1000, seed=42)
deep_metrics = compute_hedging_metrics(deep_result.pnl_samples, deep_result.cost_samples)

print("Trained Deep Hedging Results (1000 episodes):")
print(f"  Mean P&L:    {deep_metrics['mean_pnl']:>8.4f}")
print(f"  Std P&L:     {deep_metrics['std_pnl']:>8.4f}")
print(f"  Sharpe:      {deep_metrics['sharpe']:>8.3f}")
print(f"  Mean Cost:   {deep_metrics['mean_cost']:>8.4f}")

## 7. Comparing Hedging Strategies

Let's do a comprehensive comparison of all strategies:

In [ ]:
# Create agents for comparison
no_hedge_agent = NoHedgingAgent()

# Compare all strategies
comparison = compare_agents(
    agents={
        "No Hedging": no_hedge_agent,
        "Delta Hedging": delta_agent,
        "Deep Hedging": deep_agent,
    },
    env=env,
    n_episodes=1000,
    seed=123,  # Different seed for fair out-of-sample comparison
)

print(comparison.summary())

In [ ]:
# Plot P&L distributions
fig, ax = plt.subplots(figsize=(12, 6))

colors = {'No Hedging': 'gray', 'Delta Hedging': 'blue', 'Deep Hedging': 'green'}

for name, result in comparison.results.items():
    ax.hist(result.pnl_samples, bins=50, density=True, alpha=0.5,
            color=colors.get(name, 'black'), label=name, edgecolor='black')

ax.axvline(x=0, color='black', linestyle='-', linewidth=2)
ax.set_xlabel('Terminal P&L', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('P&L Distribution Comparison', fontsize=14)
ax.legend(fontsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Create a metrics comparison table
try:
    df = comparison.to_dataframe()
    display(df.round(4))
except:
    # If pandas not available, print dict
    for name, metrics in comparison.metrics.items():
        print(f"\n{name}:")
        for key, value in metrics.items():
            print(f"  {key}: {value:.4f}")

## 8. Backtesting the Trained Agent

Once a hedging agent is trained, you can evaluate it on **historical or synthetic price paths** using:

1. **BacktestEngineAdapter** — Run the agent on a price/volatility time series; get P&L, costs, and optional delta-hedge benchmark.
2. **Pipeline** `deep_hedging.backtest_agent` — Load agent (from state or use delta-hedge benchmark), build or load backtest data, run the adapter, store results in context/artifacts.

Below we use the adapter directly with our trained `deep_agent` and a short synthetic path. For production workflows (config-driven, logging, artifacts), use the pipeline — see `docs/guides/deep_hedging/backtesting_hedging_agents.md` and `examples/pipelines/run_backtest_hedging_agent.py`.

In [ ]:
# Backtest the trained deep hedging agent on a synthetic price path
from datetime import date, timedelta

try:
    from src.deep_hedging.adapters.backtesting import (
        BacktestEngineAdapter,
        BacktestConfig,
        OptionParams,
    )

    n_days = 63
    rng = np.random.default_rng(123)
    dt = 1.0 / 252.0
    mu, vol = config.risk_free_rate, config.volatility
    z = rng.standard_normal(n_days)
    log_returns = (mu - 0.5 * vol**2) * dt + vol * np.sqrt(dt) * z
    prices = config.spot_initial * np.exp(np.cumsum(np.concatenate([[0], log_returns])))
    volatilities = np.full(n_days + 1, config.volatility)
    base = date.today()
    dates = [base + timedelta(days=i) for i in range(n_days + 1)]

    option_params = OptionParams(
        strike=config.strike,
        maturity=dates[-1],
        option_type=config.option_type,
        notional=config.notional,
    )
    backtest_config = BacktestConfig(
        transaction_cost=0.001,
        maturity_days=max(1, int(config.maturity * 252)),
        option_type=config.option_type,
    )
    adapter = BacktestEngineAdapter(agent=deep_agent, config=backtest_config)
    backtest_result = adapter.run_backtest(
        prices=prices,
        volatilities=volatilities,
        dates=dates,
        option_params=option_params,
        risk_free_rate=config.risk_free_rate,
        run_benchmark=True,
    )
    print(backtest_result.summary())
except ImportError as e:
    print("BacktestEngineAdapter not available:", e)

## 9. Analysis and Discussion

### 8.1 When Does Deep Hedging Win?

Deep hedging typically outperforms delta hedging when:

1. **Transaction costs are significant** — Deep hedging learns to trade less frequently when costs are high
2. **Rebalancing is infrequent** — Delta hedging accumulates error; deep hedging adapts
3. **The risk measure is non-standard** — Deep hedging can optimise for CVaR, utility, etc.
4. **The model is misspecified** — Deep hedging can learn from data without assuming BSM

### 8.2 Limitations

1. **Training data requirements** — Need many simulated or historical paths
2. **Model capacity** — Simple MLPs may not capture complex patterns
3. **Generalisation** — May not work well in regimes not seen during training
4. **Interpretability** — Neural networks are less interpretable than delta hedging

### 8.3 Extensions

- **LSTM policies** — For path-dependent strategies (e.g., under rough volatility)
- **Multi-asset hedging** — Hedge portfolios with cross-gamma exposure
- **Different risk measures** — CVaR, entropic risk for different preferences
- **Model-free training** — Train on historical data without assuming dynamics

## 10. Exercises

Try these exercises to deepen your understanding:

1. **Change the spread**: Increase `spread_bps` to 20 or 50. How does this affect the comparison?

2. **Different risk measures**: Train with `CVaRRisk(alpha=0.95)` instead of mean-variance. Does the agent learn a different strategy?

3. **Longer maturity**: Set `maturity=1.0` (1 year). Does deep hedging provide more benefit?

4. **Put options**: Change `option_type="put"`. Is there a difference?

5. **Out-of-the-money**: Try `strike=110` (OTM call). How do the strategies compare?

## 11. Further Reading

- **Technical Reference**: `docs/reference/deep_hedging/theory.md` — PhD-level theory
- **Bühler et al. (2019)**: "Deep Hedging" — The foundational paper
- **Horvath et al. (2021)**: "Deep Hedging under Rough Volatility"
- **QuantStrata RL Framework**: `docs/reference/q_learning/rl_framework.md`